# Model Evaluation

In [1]:
import lightgbm as lgb
import polars as pl
import json
import numpy as np
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from constants import PROCESSED_DATA_FOLDER, MODEL_FOLDER, DAY_CATEGORIES

In [2]:
# Cast "Route" as ENUM type to ensure identical categorical variable encoding for train and test set
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    routes: list[str] = json.load(f)
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "r", encoding="utf-8") as f:
    target_stops: dict[str, list[int]] = json.load(f)
with open(PROCESSED_DATA_FOLDER / "stops_dict.json", "r", encoding="utf-8") as f:
    stops_dict: dict[str, str] = json.load(f)
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    target_routes: list[str] = json.load(f)
target_encoding = pl.read_csv(PROCESSED_DATA_FOLDER / "mean_travel_time_encoding.csv")
Routes_enum = pl.Enum(routes)
Days_enum = pl.Enum(DAY_CATEGORIES)

Compare with single route single segment prediction

In [3]:
naive_model = lgb.Booster(model_file=MODEL_FOLDER / "naive_model/best_lgbm_model.txt")
encoding_model = lgb.Booster(
    model_file=MODEL_FOLDER / "target_encoding_model/best_lgbm_model.txt"
)

In [4]:
def evaluate_route_naive(route: str) -> None:

    def _prepare_evaluation_data(
        route: str, depart_id: int, arrival_id: int
    ) -> tuple[np.ndarray, np.ndarray]:
        test = (
            pl.read_parquet(PROCESSED_DATA_FOLDER / "test.parquet")
            .filter(
                pl.col("Route") == route,
                pl.col("DepartureStop") == depart_id,
                pl.col("ArrivalStop") == arrival_id,
            )
            .with_columns(
                pl.col("Route").cast(Routes_enum).to_physical(),
                pl.col("DayOfWeek").cast(Days_enum).to_physical(),
            )
            .cast({"MinutesFromMidnight": pl.Float16})
        )
        return (
            test.drop("TravelDuration_min").to_numpy(),
            test["TravelDuration_min"].to_numpy(),
        )

    def _pairwise(lst):
        return list(zip(lst, lst[1:]))

    def _print_tables(route: str, depart_id: int, arrival_id: int) -> None:
        x, y = _prepare_evaluation_data(route, depart_id, arrival_id)
        output = pl.DataFrame({"prediction": naive_model.predict(x), "label": y})
        average_time = np.mean(y)
        loose_criterion = average_time * 0.1
        strict_criterion = average_time * 0.05
        residuals = (output["prediction"] - output["label"]).abs()

        print(
            f"Between {stops_dict[str(depart_id)]} and {stops_dict[str(arrival_id)]}, average travel time is {average_time:.1f} minutes"
        )
        display(
            output.select(
                (
                    (
                        abs(pl.col("prediction") - pl.col("label")) < loose_criterion
                    ).mean()
                    * 100
                )
                .round(2)
                .alias("loose_criterion (%)"),
                (
                    (
                        abs(pl.col("prediction") - pl.col("label")) < strict_criterion
                    ).mean()
                    * 100
                )
                .round(2)
                .alias("strict_criterion (%)"),
                pl.lit(residuals.quantile(0.9)).round(2).alias("p90_coverage_error"),
                pl.lit(r2_score(output["label"], output["prediction"]))
                .round(2)
                .alias("r2"),
                pl.lit(root_mean_squared_error(output["label"], output["prediction"]))
                .round(2)
                .alias("rmse"),
                pl.lit(mean_absolute_error(output["label"], output["prediction"]))
                .round(2)
                .alias("mae"),
            )
        )

    for pair in _pairwise(target_stops[route]):
        _print_tables(route, *pair)

In [5]:
def evaluate_route_target_encoding(route: str) -> None:

    def _prepare_evaluation_data(
        route: str, depart_id: int, arrival_id: int
    ) -> tuple[np.ndarray, np.ndarray]:
        test = (
            pl.read_parquet(PROCESSED_DATA_FOLDER / "test.parquet")
            .filter(
                pl.col("Route") == route,
                pl.col("DepartureStop") == depart_id,
                pl.col("ArrivalStop") == arrival_id,
            )
            .join(
                target_encoding,
                on=["Route", "DepartureStop", "ArrivalStop"],
                how="left",
            )
            .with_columns(
                pl.col("Route").cast(Routes_enum).to_physical(),
                pl.col("DayOfWeek").cast(Days_enum).to_physical(),
            )
            .cast({"MinutesFromMidnight": pl.Float16})
            .select(
                [
                    "Route",
                    "TravelDuration",
                    "MinutesFromMidnight",
                    "DayOfWeek",
                    "TravelDuration_min",
                ]
            )
        )
        return (
            test.drop("TravelDuration_min").to_numpy(),
            test["TravelDuration_min"].to_numpy(),
        )

    def _pairwise(lst):
        return list(zip(lst, lst[1:]))

    def _print_tables(route: str, depart_id: int, arrival_id: int) -> None:
        x, y = _prepare_evaluation_data(route, depart_id, arrival_id)
        output = pl.DataFrame({"prediction": encoding_model.predict(x), "label": y})
        average_time = np.mean(y)
        loose_criterion = average_time * 0.1
        strict_criterion = average_time * 0.05
        residuals = (output["prediction"] - output["label"]).abs()

        print(
            f"Between {stops_dict[str(depart_id)]} and {stops_dict[str(arrival_id)]}, average travel time is {average_time:.1f} minutes"
        )
        display(
            output.select(
                (
                    (
                        abs(pl.col("prediction") - pl.col("label")) < loose_criterion
                    ).mean()
                    * 100
                )
                .round(2)
                .alias("loose_criterion (%)"),
                (
                    (
                        abs(pl.col("prediction") - pl.col("label")) < strict_criterion
                    ).mean()
                    * 100
                )
                .round(2)
                .alias("strict_criterion (%)"),
                pl.lit(residuals.quantile(0.9)).round(2).alias("p90_coverage_error"),
                pl.lit(r2_score(output["label"], output["prediction"]))
                .round(2)
                .alias("r2"),
                pl.lit(root_mean_squared_error(output["label"], output["prediction"]))
                .round(2)
                .alias("rmse"),
                pl.lit(mean_absolute_error(output["label"], output["prediction"]))
                .round(2)
                .alias("mae"),
            )
        )

    for pair in _pairwise(target_stops[route]):
        _print_tables(route, *pair)

In [21]:
evaluate_route_naive("7500G1")
evaluate_route_target_encoding("7500G1")

Between 臺南轉運站 and 永康轉運站, average travel time is 19.9 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
50.64,27.59,4.22,0.19,2.77,2.21


Between 永康轉運站 and 麻豆站轉運站, average travel time is 18.6 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
84.25,56.04,2.37,0.45,1.68,1.13


Between 麻豆站轉運站 and 經國轉運站, average travel time is 170.7 minutes


...

loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
82.6,56.41,2.44,0.45,1.69,1.14


Between 麻豆站轉運站 and 經國轉運站, average travel time is 170.7 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
93.87,73.75,13.65,0.61,9.45,6.91


Between 經國轉運站 and 三重站, average travel time is 23.2 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
47.1,20.56,5.75,0.65,3.82,2.98


In [6]:
evaluate_route_naive("172801")
evaluate_route_target_encoding("172801")

Between 仁愛復興路口(福華飯店) and 捷運大坪林站, average travel time is 25.0 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
62.06,38.31,4.4,0.24,2.96,2.22


Between 捷運大坪林站 and 龍潭運動公園, average travel time is 50.8 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
63.7,38.29,9.31,0.42,7.16,4.85


Between 龍潭運動公園 and 新竹轉運站, average travel time is 50.3 minutes


...

loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
60.41,38.12,4.51,0.19,3.05,2.3


Between 捷運大坪林站 and 龍潭運動公園, average travel time is 50.8 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
63.7,36.48,9.3,0.38,7.39,4.97


Between 龍潭運動公園 and 新竹轉運站, average travel time is 50.3 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
70.95,39.66,9.2,0.62,6.68,4.52


In [23]:
evaluate_route_naive("560801")
evaluate_route_target_encoding("560801")

Between 新竹站 and 關東橋, average travel time is 25.3 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
59.35,34.23,5.19,0.59,3.58,2.54


Between 關東橋 and 托盤山, average travel time is 15.7 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
54.61,29.87,3.34,0.56,2.08,1.65


Between 托盤山 and 下公館站, average travel time is 12.7 minutes


...

loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
60.84,34.44,4.99,0.61,3.52,2.49


Between 關東橋 and 托盤山, average travel time is 15.7 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
52.88,28.62,3.45,0.54,2.12,1.69


Between 托盤山 and 下公館站, average travel time is 12.7 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
61.27,33.4,2.94,0.23,1.94,1.37


In [35]:
evaluate_route_naive("096801")
evaluate_route_target_encoding("096801")

Between 南竹路四段37號 and 庫倫街口, average travel time is 46.5 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
69.41,38.82,10.15,0.84,5.87,4.23


Between 南竹路四段37號 and 庫倫街口, average travel time is 46.5 minutes


loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
f64,f64,f64,f64,f64,f64
65.88,41.18,10.87,0.83,6.05,4.27
